# 🚀 Sistema RAG Completo con Groq (GRATIS y Rápido)
**Arquitectura explicada paso a paso**

**Pasos RAG:**
1. **Carga** → Lee documentos
2. **Divide** → Chunks manejables
3. **Incrusta** → Texto → vectores
4. **Almacena** → Vector DB
5. **Recupera** → Chunks relevantes
6. **Genera** → LLM + contexto

## 1. INSTALACIÓN Y CONFIGURACIÓN

In [76]:
%pip install langchain-classic langchain-groq langchain-chroma faiss-cpu python-dotenv langchain-huggingface beautifulsoup4 sentence-transformers chromadb pinecone-client cohere pymupdf -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\macdu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [77]:
from dotenv import load_dotenv

# Crea un archivo .env en la raíz del proyecto con tu API key:
#   GROQ_API_KEY=gsk_xxxxxxxxxxxx
# Obtén una clave gratuita en: https://console.groq.com/keys
# Para LangSmith (evaluación), añade también:
#   LANGSMITH_API_KEY=lsv2_xxxxxxxxxxxx  → https://smith.langchain.com
#   LANGSMITH_PROJECT=mi-rag-groq
load_dotenv()
print('✅ .env cargado')

✅ .env cargado


## 2. CARGA DE DOCUMENTOS

In [78]:
from langchain_community.document_loaders import WebBaseLoader, PyMuPDFLoader
import json
import re

def clean_web_doc(doc):
    text = doc.page_content
    text = re.sub(r'\n{3,}', '\n\n', text)      # colapsa saltos de línea múltiples
    text = re.sub(r'[ \t]{2,}', ' ', text)       # colapsa espacios y tabulaciones
    doc.page_content = text.strip()
    return doc

def clean_pdf_doc(doc):
    text = doc.page_content
    text = re.sub(r'-\n', '', text)              # une palabras cortadas con guión
    text = re.sub(r'\n{3,}', '\n\n', text)       # colapsa saltos múltiples
    text = re.sub(r'[ \t]{2,}', ' ', text)       # colapsa espacios y tabulaciones
    text = re.sub(r'<[^>]+>', '', text)           # elimina etiquetas HTML residuales
    text = re.sub(r'\(cid:\d+\)', '', text)       # elimina referencias a imágenes de PyMuPDF
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)   # elimina caracteres no ASCII (iconos, símbolos raros)
    doc.page_content = text.strip()
    return doc

# Leer URLs desde fichero externo
with open("urls.json", "r", encoding="utf-8") as f:
    data = json.load(f)

urls = data["news"]
loader = WebBaseLoader(urls)
url_docs = loader.load()
url_docs = [clean_web_doc(d) for d in url_docs]
for doc in url_docs:
    doc.metadata["source_type"] = "web"
print(f"📄 Cargados {len(url_docs)} documentos web")

# --- PDF ---
pdf_loader = PyMuPDFLoader("organic.pdf")
pdf_docs = pdf_loader.load()
pdf_docs = [clean_pdf_doc(d) for d in pdf_docs]
for doc in pdf_docs:
    doc.metadata["source_type"] = "pdf"
print(f"📄 Cargados {len(pdf_docs)} documentos PDF")

docs = url_docs + pdf_docs
print(f"📄 Cargados y limpios {len(docs)} documentos totales")

📄 Cargados 4 documentos web
📄 Cargados 4 documentos PDF
📄 Cargados y limpios 8 documentos totales


## 3. DIVISIÓN EN FRAGMENTOS (CHUNKS)

In [79]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Divide documentos largos en chunks que quepan en el contexto del LLM.
# chunk_size=1000   → ~750 tokens por chunk (margen de seguridad)
# chunk_overlap=200 → los chunks se solapan 200 chars para no perder
#                     contexto en los cortes (ej: una frase a caballo)
# separators        → corta preferentemente en párrafos, luego líneas,
#                     luego frases; nunca a mitad de palabra
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?"]
)
splits = text_splitter.split_documents(docs)
print(f"✂️ Divididos en {len(splits)} chunks")
print(f"Ejemplo chunk: {len(splits[0].page_content)} chars\n{splits[0].page_content[:200]}...")

✂️ Divididos en 73 chunks
Ejemplo chunk: 743 chars
Trump dice ahora que no descarta desplegar soldados en Irán: "Me dan igual las encuestas" | Internacional

Es noticiaÚltimas noticiasGuerra IránSánchez TrumpMargarita RoblesChinaPasarela Bocal Santand...


## 4. EMBEDDINGS + VECTOR STORE

In [80]:
from langchain_huggingface import HuggingFaceEmbeddings

# paraphrase-multilingual-MiniLM-L12-v2: obligatorio cuando mezclas idiomas.
# Español e inglés comparten el mismo espacio vectorial, así las distancias
# entre chunks son comparables independientemente del idioma del documento.
# ⚠️ Si tenías un chroma_db generado con otro modelo, bórralo antes de ejecutar:
#    import shutil; shutil.rmtree('./chroma_db', ignore_errors=True)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("✅ Embeddings multilingües listos", embeddings)

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None)), '(Request ID: 5e7cb445-9d17-48fa-ad66-b735c7b8ee9f)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


✅ Embeddings multilingües listos model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [81]:
from langchain_community.vectorstores import FAISS
# FAISS indexa todos los vectores para hacer búsqueda por similitud coseno, y no es persistente (en memoria), 
# pero es muy rápido para prototipos y pruebas locales, con búsquedas en ~1ms, independientemente del número de chunks.
# Para producción multi-usuario considera Chroma (local) o Pinecone (nube).
vectorstore_faiss = FAISS.from_documents(documents=splits, embedding=embeddings)
print("✅ Vectorstore listo (384-dim vectores indexados)")
print(f"   Chunks indexados: {vectorstore_faiss.index.ntotal}")

✅ Vectorstore listo (384-dim vectores indexados)
   Chunks indexados: 73


In [82]:
from langchain_chroma import Chroma
import hashlib

def get_document_hash(document):
    return hashlib.md5(document.page_content.encode('utf-8')).hexdigest()

# ⚠️ Si cambias los documentos fuente, borra chroma_db antes de reejecutar:
#    import shutil; shutil.rmtree('./chroma_db', ignore_errors=True)
vectorstore_chroma = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

ids = [get_document_hash(doc) for doc in splits]
vectorstore_chroma.add_documents(documents=splits, ids=ids)
print(f"   Chunks total en Chroma: {vectorstore_chroma._collection.count()}")

# Desglose por fuente — útil para detectar desbalanceo
from collections import Counter
all_metas = vectorstore_chroma._collection.get(include=['metadatas'])['metadatas']
counts = Counter(m.get('source_type', 'unknown') for m in all_metas)
print("   Desglose por tipo de fuente:", dict(counts))

   Chunks total en Chroma: 148
   Desglose por tipo de fuente: {'web': 115, 'pdf': 33}


## 5. LLM + CADENA RAG

In [83]:
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate

def llm_rag_chain(vectorstore):
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.1
    )

    prompt_template = """Usa SOLO información de los documentos proporcionados.
    Si no encuentras respuesta, di 'No tengo información suficiente'.

    CONTEXTO:
    {context}

    PREGUNTA: {question}
    RESPUESTA CONCISA:"""
    PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 4}
        ),
        chain_type_kwargs={"prompt": PROMPT},
        return_source_documents=True
    )
    print("🔥 Cadena RAG lista!")
    return qa_chain


In [84]:
qa_chain_faiss = llm_rag_chain(vectorstore_faiss)

🔥 Cadena RAG lista!


In [85]:
qa_chain_chroma = llm_rag_chain(vectorstore_chroma)

🔥 Cadena RAG lista!


## 6. ¡PRUEBAS RAG! 🔥

In [86]:
def rag_test(qa_chain, text_query):
    resultado = qa_chain.invoke({"query": text_query})
    print("🤖 RESPUESTA:")
    print(resultado["result"])
    print("\n" + "="*60)
    print("📄 FUENTES (top-4):")
    for i, doc in enumerate(resultado["source_documents"][:4]):
        print(f"{i+1}. [{doc.metadata.get('source_type', '?').upper()}] {doc.metadata.get('source', doc.metadata.get('file_path', 'Desconocido'))[:60]}")
        print(f"   Preview: {doc.page_content[:150]}...\n")


In [87]:
consulta = "¿Me puedes hablar de los productos organicos y alguna irregularidad que se mencione y se realice?"
rag_test(qa_chain_faiss, consulta)

🤖 RESPUESTA:
Según la información proporcionada, los productos orgánicos deben cumplir con ciertas regulaciones y etiquetado específico. Algunas irregularidades que se mencionan incluyen:

* No identificar separadamente los ingredientes orgánicos y no orgánicos en la declaración de ingredientes.
* Utilizar el sello de USDA orgánico en productos que no son 100% orgánicos.
* Representar un producto como orgánico cuando no lo es.
* No incluir la declaración "certificado orgánico por" en la etiqueta.
* No cumplir con los requisitos de tamaño y formato de la letra en las declaraciones de "hecho con orgánico" y porcentaje de ingredientes orgánicos.

Es importante destacar que los productos que se etiquetan como "hecho con orgánico" deben cumplir con ciertos requisitos, como incluir al menos un 70% de ingredientes orgánicos y no utilizar el sello de USDA orgánico.

📄 FUENTES (top-4):
1. [PDF] organic.pdf
   Preview: 3. Organic and nonorganic forms of the same ingredient; 
4. Percentage of org

In [88]:
consulta = "What does Trump think about Iran?"
rag_test(qa_chain_chroma, consulta)

🤖 RESPUESTA:
Trump no descarta el envío de tropas a Irán "si son necesarias" y afirma que "los estamos destrozando". También anuncia que una "gran ola" de ataques contra Irán "está a punto de llegar".

📄 FUENTES (top-4):
1. [WEB] https://theobjective.com/internacional/2026-03-02/trump-envi
   Preview: Internacional

 
                            Trump no descarta el envío de tropas a Irán «si son necesarias»: «Los estamos destrozando»               ...

2. [WEB] https://theobjective.com/internacional/2026-03-02/trump-envi
   Preview: Internacional

 
 Trump no descarta el envío de tropas a Irán «si son necesarias»: «Los estamos destrozando» 
El presidente de EEUU anuncia que la «gr...

3. [WEB] https://www.elperiodico.com/es/internacional/20260224/trump-
   Preview: DirectoAzerbaiyán denuncia que han caído en su territorio drones lanzados desde Irán Intervención ante las CámarasTrump amenaza a Irán y se enfrenta a...

4. [WEB] https://www.elperiodico.com/es/internacional/20260224/trump

## 7. GUARDAR/RECARGAR VECTORSTORE (PRODUCCIÓN)

In [89]:
# Persiste el índice en disco: genera index.faiss (vectores) + index.pkl (metadatos).
# Así evitas re-embeber los documentos cada vez que reinicias el notebook.
vectorstore_faiss.save_local("mi_rag_index")

# allow_dangerous_deserialization=True es requerido porque FAISS usa pickle.
# Solo carga índices que hayas generado tú mismo; nunca de fuentes externas.
vectorstore_reloaded = FAISS.load_local(
    "mi_rag_index", embeddings, allow_dangerous_deserialization=True
)
print("💾 Index guardado y recargado correctamente")
print(f"   Chunks en el índice: {vectorstore_reloaded.index.ntotal}")

💾 Index guardado y recargado correctamente
   Chunks en el índice: 73


In [90]:
# Chroma: Ya está persistente automáticamente. 
# No es necesario usar save/load explícitamente, ni descomentar el bloque de código, 
# ya que de lo contrario estarías duplicando los chunks (ya que se cargan al reiniciar el entorno).
# 
# Si se descomenta, el número de chunks aumentaría innecesariamente.
# 
# vectorstore_chroma_reloaded = Chroma(
#     persist_directory="./chroma_db",
#     embedding_function=embeddings
# )
# print("💾 Chroma recargado (ya persistente)")
# print(f"   Chunks: {vectorstore_chroma_reloaded._collection.count()}")

## 🎯 PRÓXIMOS PASOS

| Mejora              | Cómo                        | Beneficio                                        | Estado |
|---------------------|-----------------------------|--------------------------------------------------|--------|
| **Streaming**        | `qa_chain.stream()`          | Respuestas token a token en tiempo real         | -      |
| **Reranking**        | `CohereRerank`               | Reordena los chunks recuperados → precisión +30% | -      |
| **Cloud DB**         | `Chroma` / `Pinecone`        | Persistencia escalable y multi-usuario           | ✅     |
| **PDFs**             | `PyMuPDFLoader`              | Ingesta documentos empresariales                 | ✅     |
| **Multi-idioma**     | `paraphrase-multilingual-MiniLM-L12-v2` | Embeddings nativos en español y otros idiomas | ✅     |
| **Evaluación con LangSmith** | `langsmith` + `evaluate()` | Mide la calidad de la cadena RAG con datasets de preguntas/respuestas esperadas... | - |
